# IHGAMP - Step 2: Feature Extraction

Extract embeddings using pathology foundation models:
- UNI (ViT-L/16, 1024-d)
- OpenSlideFM (ViT-B/16, 768-d)

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import torch

from src.embeddings import EmbeddingExtractor, EmbeddingConfig
from src.preprocessing import TileExtractor
from src.utils import extract_patient_id

OUTPUT_DIR = Path('./outputs')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Load manifest
manifest = pd.read_csv(OUTPUT_DIR / 'slide_manifest.csv')
print(f'Loaded {len(manifest)} slides')

In [ ]:
# Initialize extractors
embed_config = EmbeddingConfig(
    model_name='UNI',
    embedding_dim=1024,
    batch_size=64,
    device=DEVICE,
    precision='fp16'
)
extractor = EmbeddingExtractor(embed_config)
tile_extractor = TileExtractor(patch_size=256, patch_cap=2048)

In [ ]:
# Extract embeddings
EMBEDDINGS_DIR = OUTPUT_DIR / 'embeddings'
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

embeddings_df = extractor.process_slides(
    slide_paths=manifest['slide_path'].tolist(),
    output_path=str(EMBEDDINGS_DIR / 'slide_embeddings.parquet'),
    tile_extractor=tile_extractor
)
print(f'Extracted embeddings for {len(embeddings_df)} slides')

In [ ]:
# Aggregate to patient level
embeddings_df['patient'] = embeddings_df['slide'].apply(
    lambda x: extract_patient_id(Path(x).stem)
)
z_cols = [c for c in embeddings_df.columns if c.startswith('z')]
patient_embeddings = embeddings_df.groupby('patient')[z_cols].mean().reset_index()
patient_embeddings.to_parquet(EMBEDDINGS_DIR / 'patient_embeddings.parquet', index=False)
print(f'Patient-level embeddings: {len(patient_embeddings)}')